
# The Multilayer Perceptron (MLP)

## From One to Many

A single Perceptron is limited to linear decisions. To solve complex problems, we connect many neurons together in layers. This structure is called a **Multilayer Perceptron (MLP)** or a **Feedforward Neural Network**.

## Architecture

An MLP consists of at least three layers:

1.  **Input Layer**: Receives the raw features vector $X$. (No computation happens here).
2.  **Hidden Layers**: Layers between input and output. They perform the computation and extract "features" from the data. The more hidden layers, the "Deep" the network (Deep Learning).
3.  **Output Layer**: Produces the final prediction (e.g., probability of "Cat").

## Activation Functions

Without activation functions, an MLP is just a giant Linear Regression model (because linear + linear = linear). **Activation Functions** introduce **Non-Linearity**, allowing the network to learn curves and complex shapes.

| Function    | Formula                             | Range         | Usage                                                          |
|----------- |----------------------------------- |------------- |-------------------------------------------------------------- |
| **Sigmoid** | $\frac{1}{1+e^{-z}}$                | $(0, 1)$      | Output layer (Binary Classification).                          |
| **Tanh**    | $\frac{e^z - e^{-z}}{e^z + e^{-z}}$ | $(-1, 1)$     | Hidden layers (Legacy). Zero-centered.                         |
| **ReLU**    | $\max(0, z)$                        | $[0, \infty)$ | **Standard for Hidden Layers**. Fast and efficient.            |
| **Softmax** | $\frac{e^{z_i}}{\sum e^{z_j}}$      | $(0, 1)$      | Output layer (Multiclass). Returns probabilities summing to 1. |

## Forward Propagation (The Math)

How does the data flow from Input to Output? It is a series of **Matrix Multiplications**.

For a single layer $l$:

1.  **Linear Step ($Z$)**: Weighted sum of inputs from previous layer ($A^{[l-1]}$). $$Z^{[l]} = W^{[l]} \cdot A^{[l-1]} + b^{[l]}$$
2.  **Activation Step ($A$)**: Apply the function. $$A^{[l]} = g(Z^{[l]})$$

This process repeats until the output layer.

## Practical Demonstration: Anatomy of an MLP

We will first visualize the activation functions, then build a "Forward Pass" from scratch using NumPy to understand the matrix math.

### Visualizing Activation Functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Define functions
def sigmoid(z): return 1 / (1 + np.exp(-z))
def tanh(z): return np.tanh(z)
def relu(z): return np.maximum(0, z)

# Generate range
z = np.linspace(-5, 5, 100)

# Plot
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.plot(z, sigmoid(z), color='navy')
plt.title("Sigmoid (0 to 1)")
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(z, tanh(z), color='purple')
plt.title("Tanh (-1 to 1)")
plt.grid(True)

plt.subplot(1, 3, 3)
plt.plot(z, relu(z), color='crimson')
plt.title("ReLU (0 to ∞)")
plt.grid(True)

plt.show()

### Forward Propagation from Scratch

Let's manually calculate the output of a tiny network:

-   **Input**: 3 features.
-   **Hidden Layer**: 4 neurons (ReLU).
-   **Output Layer**: 1 neuron (Sigmoid).

In [ ]:
np.random.seed(42)

# 1. Inputs (Batch of 2 samples, 3 features each)
X = np.array([[0.5, 0.1, -0.2],
              [0.9, -0.5, 0.1]])
print(f"Input Shape: {X.shape}")

# 2. Hidden Layer (Weights: 3 inputs -> 4 neurons)
W1 = np.random.randn(3, 4)
b1 = np.zeros((4,))

# Linear Step Z1 = X dot W + b
Z1 = np.dot(X, W1) + b1
# Activation A1 = ReLU(Z1)
A1 = np.maximum(0, Z1)

print(f"Hidden Activation Shape: {A1.shape}")

# 3. Output Layer (Weights: 4 inputs -> 1 neuron)
W2 = np.random.randn(4, 1)
b2 = np.zeros((1,))

# Linear Step Z2
Z2 = np.dot(A1, W2) + b2
# Activation A2 = Sigmoid(Z2)
output = 1 / (1 + np.exp(-Z2))

print(f"Output Shape: {output.shape}")
print(f"Predictions:\n{output}")

**Insight**: The magic of Neural Networks is that \`np.dot\` (Matrix Multiplication) handles the connections for us automatically.

## Practical Demonstration: Sklearn MLPClassifier

Now we use `scikit-learn` to solve the non-linear "Moons" dataset using an MLP.

### Generate Data

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X, y = make_moons(n_samples=500, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

plt.figure(figsize=(6, 5))
plt.scatter(X[:,0], X[:,1], c=y, cmap='coolwarm', edgecolors='k')
plt.title("Moons Dataset (Non-Linear)")
plt.show()

### Train MLP

We use 2 hidden layers with 10 neurons each.

-   `hidden_layer_sizes=(10, 10)`
-   `activation='relu'`

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.inspection import DecisionBoundaryDisplay

# Initialize
mlp = MLPClassifier(hidden_layer_sizes=(10, 10), 
                    activation='relu', 
                    solver='adam', 
                    max_iter=1000, 
                    random_state=42)

# Train
mlp.fit(X_train, y_train)

print(f"Training Accuracy: {mlp.score(X_train, y_train):.2f}")
print(f"Test Accuracy:     {mlp.score(X_test, y_test):.2f}")

### Visualize the "Neural" Boundary

In [ ]:
plt.figure(figsize=(8, 6))
DecisionBoundaryDisplay.from_estimator(mlp, X, cmap="coolwarm", alpha=0.8)
plt.scatter(X[:,0], X[:,1], c=y, cmap="coolwarm", edgecolors='k')
plt.title("MLP Decision Boundary (ReLU)")
plt.show()

## Exercises

### Activation Function Impact

Train an MLP using `activation='tanh'` and another using `activation='logistic'` (Sigmoid).

-   Compare the decision boundaries.
-   **Note**: Tanh often creates smoother curves than ReLU (which creates piecewise linear shapes).

### Network Capacity (Width vs Depth)

Compare two architectures:

1.  **Wide**: 1 Hidden Layer with 100 neurons `(100,)`.
2.  **Deep**: 5 Hidden Layers with 10 neurons each `(10, 10, 10, 10, 10)`.

Do they perform differently?

## Summary

1.  **Layers**: Input (Passive) $\rightarrow$ Hidden (Active Feature Extraction) $\rightarrow$ Output (Prediction).
2.  **Activations**:
    -   **ReLU**: The default for hidden layers.
    -   **Sigmoid/Softmax**: The default for the output layer.
3.  **Forward Prop**: Data moves through the network via matrix multiplication ($Z=WX+b$) and activation functions ($A=f(Z)$).